In [ ]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)  # Load credentials from the project .env file.

google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError(
        "Set GOOGLE_API_KEY in .env to a valid Google AI Studio API key; "
        "do not use an OAuth access token."
    )

In [ ]:
class SubState(TypedDict):
    input_text: str
    translated_text: str


In [ ]:
primary = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=google_api_key,
    max_retries=3,
)
backup = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=google_api_key,
    max_retries=3,
)

model = primary.with_fallbacks([backup])

subgraph_llm = model

In [ ]:
def translate_text(state: SubState):

    prompt = f"""
Translate the following text to Hindi.
Keep it natural and clear. Do not add extra content.

Text:
{state["input_text"]}
""".strip()
    
    translated_text = subgraph_llm.invoke(prompt).content

    return {'translated_text': translated_text}


In [ ]:
subgraph = StateGraph(SubState)
subgraph.add_node("translate_text",translate_text)
subgraph.add_edge(START,"translate_text")
subgraph.add_edge("translate_text",END)

In [ ]:
subgraph = subgraph.compile()

In [ ]:
class ParentState(TypedDict):
    question: str
    answer_eng: str
    answer_hin: str


In [ ]:
parent_llm = model

In [ ]:
def generate_answer(state: ParentState):
    answer = parent_llm.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng': answer}


In [ ]:
def translate_answer(state: ParentState):

    # call the subgraph
    result = subgraph.invoke({'input_text': state['answer_eng']})

    return {'answer_hin': result['translated_text']}


In [ ]:
parent_builder = StateGraph(ParentState)
parent_builder.add_node('generate_answer',generate_answer)
parent_builder.add_node('translate_answer',translate_answer)

parent_builder.add_edge(START,'generate_answer')
parent_builder.add_edge('generate_answer','translate_answer')
parent_builder.add_edge('translate_answer',END)

In [ ]:
graph = parent_builder.compile()

In [ ]:
graph

In [ ]:
graph.invoke({'question': 'What is quantum physics'})